In [1]:
!pip install -q "transformers==4.55.4" accelerate jinja2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-4.0-1.2B"  # 허브 저장소 식별자
tokenizer = AutoTokenizer.from_pretrained(model_name)  # 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # T4 GPU용 16비트 자료형
    device_map="auto",         # 사용 가능한 GPU에 자동 배치
)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

In [3]:
prompt = "한국의 봄을 한 문장으로 설명해 줘."
messages = [
    {"role": "user", "content": prompt}
]

In [4]:
raw_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # 질문만 토큰화

with torch.inference_mode():
    raw_output_ids = model.generate(
        **raw_inputs,
        max_new_tokens=40,  # 새로 생성할 토큰 수의 상한
        do_sample=False,    # 무작위 추출을 사용하지 않음
    )

raw_length = raw_inputs["input_ids"].shape[1]
tokenizer.decode(raw_output_ids[0, raw_length:])  # 특수 토큰을 포함해 복원


'[|endofturn|]'

In [5]:
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,              # 변환 결과를 문자열로 반환
    add_generation_prompt=True,  # 어시스턴트 응답 차례 추가
    enable_thinking=False,       # 일반 응답 형식 사용
)
formatted_prompt


'[|user|]\n한국의 봄을 한 문장으로 설명해 줘.[|endofturn|]\n[|assistant|]\n<think>\n\n</think>\n\n'

In [6]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,               # 변환 결과를 토큰 ID로 반환
    add_generation_prompt=True,  # 어시스턴트 응답 차례 추가
    enable_thinking=False,       # 일반 응답 형식 사용
    return_dict=True,            # input_ids와 attention_mask를 사전으로 반환
    return_tensors="pt",         # 사전의 값을 파이토치 텐서로 변환
).to(model.device)               # 입력을 모형과 같은 장치로 이동

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40,  # 새로 생성할 토큰 수의 상한
        do_sample=False,    # 무작위 추출을 사용하지 않음
        use_cache=True,     # 이전 토큰의 Key·Value를 KV 캐시에 저장해 재사용
    )


In [7]:
input_length = inputs["input_ids"].shape[1]
generated_ids = output_ids[0, input_length:]
tokenizer.decode(generated_ids, skip_special_tokens=True)

'"봄은 따뜻한 바람과 함께 꽃들이 피며 새로운 시작을 맞이하는 계절입니다."'

In [8]:
from google.colab import userdata

openrouter_api_key = userdata.get("OPENROUTER_API_KEY")

In [9]:
!pip install -q openai


In [10]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)


In [11]:
messages = [
    {
        "role": "user",
        "content": "생성형 AI가 교육에 활용되는 예를 두 문장으로 설명해 줘.",
    }
]

response = client.chat.completions.create(
    model="openrouter/free",
    messages=messages,
    max_tokens=120,  # 응답 길이의 상한
    temperature=0.7,
)


In [12]:
response.model  # 실제로 응답한 모형


'nvidia/nemotron-3-super-120b-a12b:free'

In [13]:
response.choices[0].message.content  # 첫 번째 응답의 내용


'생성형 AI는 학생 개개인의 학습 수준에 맞춘 맞춤형 문제와 해설을 자동 생성하여 개인별 학습을 지원합니다. 또한, 교사는 AI가 만든 시뮬레이션 사례나 토론 주제를 활용해 실제 상황 기반의 프로젝트 학습을 설계할 수 있습니다.'

In [14]:
response = client.chat.completions.create(
    model="google/gemini-2.5-flash-lite",
    messages=messages,
    max_tokens=120,
    temperature=0.7,
)

response.choices[0].message.content  # 생성된 텍스트


'생성형 AI는 학생 개개인의 학습 수준과 관심사에 맞춰 개인화된 학습 자료를 만들거나, 복잡한 개념을 쉽게 설명하는 데 활용될 수 있습니다. 또한, 학생들이 작성한 글이나 코드를 분석하고 개선점을 제시하여 학습 효과를 높이는 데 도움을 줄 수 있습니다.'

In [15]:
model_names = [
    "google/gemini-2.5-flash-lite",
    "openai/gpt-4.1-mini",
]

for model_name in model_names:
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=120,
        temperature=0.7,
    )
    print(response.model)
    print(response.choices[0].message.content)


google/gemini-2.5-flash-lite
생성형 AI는 학생들의 학습 수준과 관심사에 맞춰 개인화된 학습 자료를 만들거나, 복잡한 개념을 쉽게 설명하는 튜터 역할을 수행하여 교육 경험을 풍부하게 할 수 있습니다. 또한, 교사들에게는 수업 계획 수립, 평가 문항 생성, 학생 피드백 작성 등 행정 업무 부담을 줄여주어 교육의 질을 향상시키는 데 기여합니다.
openai/gpt-4.1-mini
생성형 AI는 학생 맞춤형 학습 자료를 자동으로 제작하여 개별 학습 효과를 높인다. 또한, 교사는 AI가 생성한 예제 문제나 피드백을 활용해 수업 준비 시간을 절약할 수 있다.


In [16]:
response.usage.prompt_tokens  # 입력 토큰 수


27

In [17]:
response.usage.completion_tokens  # 생성 토큰 수


55

In [18]:
response.usage.total_tokens  # 전체 토큰 수


82

In [21]:
text = "오늘 날씨가 맑다." # 예시 문장
inputs = tokenizer(text, return_tensors="pt").to(model.device) # 토큰화 하여 PyTorch 텐서로 변환
with torch.no_grad(): # 기울기 계산을 생략
    outputs = model(**inputs, labels=inputs["input_ids"]) # 다음 토큰 예측 손실 계산
outputs.loss  # 평균 교차 엔트로피 손실


tensor(6.6681, device='cuda:0')

In [22]:
torch.exp(outputs.loss).item()  # 혼란도


786.8629760742188

In [23]:
!pip install -q nltk


In [29]:
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

reference = """the cat is on the mat""".strip().split()

hypothesis = """the cat sits on the mat""".strip().split()


In [30]:
sentence_bleu([reference], hypothesis)


7.262123179505913e-78

In [32]:
sentence_bleu([reference], hypothesis, weights=(1 / 3, 1 / 3, 1 / 3))


0.5

In [27]:
sentence_bleu(
    [reference],
    hypothesis,
    smoothing_function=SmoothingFunction().method2,
)


0.18353610946423723